# Generate MultiMNIST Training Dataset

## Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from matplotlib.patches import Rectangle

SEED = 42

TARGET_NUM_IMAGES = 200000
# Total number of generated images.

SOURCE_DATA = "./data/train.pt"
OUTPUT_DATA = "./data/train_generated.pt"
MNIST_ROOT = "./mnist"

IMAGE_SIZE = 64
DIGIT_SIZE = 28

rng = np.random.default_rng(SEED)

## Download MNIST

In [ ]:
# The official torchvision MNIST dataset exposes the training images and targets.
mnist = torchvision.datasets.MNIST(
    root=MNIST_ROOT,
    train=True,
    download=True,
)

mnist_images = mnist.data.numpy().astype(np.uint8)
mnist_labels = mnist.targets.numpy().astype(np.int64)

mnist_by_class = {
    class_id: mnist_images[mnist_labels == class_id]
    for class_id in range(10)
}

assert all(len(mnist_by_class[class_id]) > 0 for class_id in range(10))
print(f"Loaded {len(mnist_images):,} MNIST training digits")

## Create Shuffled Queues

In [ ]:
queues = {}
pointers = {}

for class_id in range(10):
    queue = np.arange(len(mnist_by_class[class_id]))
    rng.shuffle(queue)
    queues[class_id] = queue
    pointers[class_id] = 0

def get_digit(class_id):
    """Return one digit, reshuffling only after this class queue is exhausted."""
    if pointers[class_id] == len(queues[class_id]):
        rng.shuffle(queues[class_id])
        pointers[class_id] = 0

    source_index = queues[class_id][pointers[class_id]]
    pointers[class_id] += 1
    return mnist_by_class[class_id][source_index]

## Load Dataset Metadata

In [ ]:
# Only all_positions is reused from train.pt. Its images and labels are not used.
source_data = torch.load(SOURCE_DATA, map_location="cpu")
source_positions = source_data["all_positions"].numpy().astype(np.int64)

assert source_positions.ndim == 3
assert source_positions.shape[-1] == 2
assert source_positions.shape[1] >= 1
assert np.all((source_positions >= -1) & (source_positions < IMAGE_SIZE))
print(f"Loaded {len(source_positions):,} position templates")

## Generate Images and Metadata

In [ ]:
num_slots = source_positions.shape[1]
template_indices = rng.integers(0, len(source_positions), size=TARGET_NUM_IMAGES)
all_positions = source_positions[template_indices].copy()

images = np.zeros((TARGET_NUM_IMAGES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
all_digit_labels = np.full((TARGET_NUM_IMAGES, num_slots), -1, dtype=np.int64)
num_digits = np.zeros(TARGET_NUM_IMAGES, dtype=np.int64)

for image_index, positions in enumerate(all_positions):
    valid_slots = np.all(positions >= 0, axis=1)
    valid_positions = positions[valid_slots]
    num_digits[image_index] = len(valid_positions)

    for slot_index, (x, y) in zip(np.flatnonzero(valid_slots), valid_positions):
        # integers in [0, 10) are uniform, so every class has probability 10%.
        digit_class = int(rng.integers(0, 10))
        all_digit_labels[image_index, slot_index] = digit_class
        digit = get_digit(digit_class)

        canvas_patch = images[image_index, y:y + DIGIT_SIZE, x:x + DIGIT_SIZE]
        mask = digit > 0
        canvas_patch[mask] = digit[mask]

assert np.all(num_digits > 0)

# center_labels is required for compatibility. Use the label at the valid
# slot whose 28x28 box center is closest to the image center.
center_labels = np.empty(TARGET_NUM_IMAGES, dtype=np.int64)
image_center = (IMAGE_SIZE - 1) / 2.0
for image_index, positions in enumerate(all_positions):
    valid_slots = np.flatnonzero(np.all(positions >= 0, axis=1))
    centers = positions[valid_slots] + DIGIT_SIZE / 2.0
    distances = np.sum((centers - image_center) ** 2, axis=1)
    center_labels[image_index] = all_digit_labels[image_index, valid_slots[np.argmin(distances)]]

count_labels = np.stack(
    [(all_digit_labels == digit).sum(axis=1) for digit in range(10)],
    axis=1,
).astype(np.int64)
labels = (count_labels > 0).astype(np.float32)
label_type = "multi_label"

print(f"Generated {len(images):,} images")

## Verification

In [ ]:
expected_count_labels = np.stack(
    [(all_digit_labels == digit).sum(axis=1) for digit in range(10)],
    axis=1,
).astype(np.int64)

assert np.array_equal(count_labels, expected_count_labels)
assert np.array_equal(labels, count_labels > 0)
assert images.dtype == np.uint8
assert labels.shape == (TARGET_NUM_IMAGES, 10)
assert count_labels.shape == (TARGET_NUM_IMAGES, 10)
assert all_digit_labels.shape == all_positions.shape
assert np.all(all_digit_labels[all_positions[..., 0] < 0] == -1)
assert np.all(all_digit_labels[all_positions[..., 1] < 0] == -1)

print("All label and metadata checks passed.")

## Visualization

In [ ]:
def visualize_samples(num_samples=16):
    indices = rng.choice(len(images), size=min(num_samples, len(images)), replace=False)
    columns = 4
    rows = int(np.ceil(len(indices) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(14, 3.5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, index in zip(axes, indices):
        ax.imshow(images[index], cmap="gray", vmin=0, vmax=255)
        for digit, (x, y) in zip(all_digit_labels[index], all_positions[index]):
            if digit < 0:
                continue
            ax.add_patch(Rectangle((x, y), DIGIT_SIZE, DIGIT_SIZE, fill=False, edgecolor="red", linewidth=1.5))
            ax.text(x + 1, y + 1, str(digit), color="yellow", fontsize=9, weight="bold")
        ax.set_title(
            f"index={index}\\n"
            f"counts={count_labels[index].tolist()}\\n"
            f"labels={all_digit_labels[index].tolist()}",
            fontsize=8,
        )
        ax.axis("off")

    for ax in axes[len(indices):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

visualize_samples()

## Statistics

In [ ]:
print(f"Number of images: {len(images):,}")
print("Digit frequencies:")
for digit, frequency in enumerate(labels.sum(axis=0).astype(np.int64)):
    print(f"  Digit {digit}: {frequency:,}")

print(f"Positive labels: {int(labels.sum()):,}")
print("Distribution of num_digits:")
values, frequencies = np.unique(num_digits, return_counts=True)
for value, frequency in zip(values, frequencies):
    print(f"  {value} digits: {frequency:,}")

source_num_digits = np.sum(np.all(source_positions >= 0, axis=-1), axis=1)
print("Position-template num_digits distribution:")
source_values, source_frequencies = np.unique(source_num_digits, return_counts=True)
for value, frequency in zip(source_values, source_frequencies):
    print(f"  {value} digits: {frequency:,}")

print(f"Image mean: {images.mean():.4f}")
print(f"Image std:  {images.std():.4f}")

## Save Dataset

In [ ]:
output = {
    "images": torch.from_numpy(images),
    "labels": torch.from_numpy(labels),
    "center_labels": torch.from_numpy(center_labels),
    "count_labels": torch.from_numpy(count_labels),
    "num_digits": torch.from_numpy(num_digits),
    "all_digit_labels": torch.from_numpy(all_digit_labels),
    "all_positions": torch.from_numpy(all_positions),
    "label_type": label_type,
}
torch.save(output, OUTPUT_DATA)
print(f"Saved dataset to {OUTPUT_DATA}")